# Keystone Project — Week 4: Advanced Visualizations + Visual Integrity
**Dataset:** Most Streamed Spotify Songs 2023  
**Source:** Nidula Elgiriyewithana, Kaggle (DOI: 10.34740/KAGGLE/DSV/6367938)  
**Download:** https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023

---

## Overview

This notebook covers the Week 4 Keystone deliverables:

1. **Ethics lens** — examining bias, representation gaps, and what this data can and cannot tell us  
2. **Revised EDA visuals** — polished, intentional charts that apply this week's design principles  
3. **Polished story visuals** — a small set of final-quality charts, each with a written explanation  
4. **TODOs / repo notes** — ideas and cleanup flags for future iterations

---


## 1. Setup & Data Load

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np


df = pd.read_csv("spotify-2023.csv", encoding="latin-1")

# ── Basic cleaning ──────────────────────────────────────────────────────────────
df["streams"] = pd.to_numeric(df["streams"], errors="coerce")

df = df.dropna(subset=["streams"]).copy()

df["streams_B"] = df["streams"] / 1e9

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head(3)


## 2. Re-examining the Data Through an Ethics Lens

Before building any visuals, pause and apply the Week 4 critical questions.

---

### What this dataset is — and isn't

This dataset contains the **most-streamed songs on Spotify in 2023**, not a random or representative sample of all music. That distinction matters for every conclusion we draw.

---

### Bias & collection concerns

**1. Survivorship bias**  
Every row in this dataset is already a winner, a track that crossed some threshold of streaming success. We cannot use this data to study what makes songs *fail* to gain traction, or what separates a mid-tier track from a top-streamed one. Any conclusion about "what popular songs have in common" is drawn from a pre-filtered population.

**2. Playlist placement ≠ organic popularity**  
`in_spotify_playlists` is one of the strongest predictors of stream counts — but playlist curation is not neutral. Major label artists have documented advantages in securing Spotify editorial playlist placement. A chart showing "playlist count vs. streams" could accidentally look like a story about audio quality when it's partly a story about industry relationships.

**3. Streams ≠ listeners**  
One superfan streaming a track 1,000 times counts identically to 1,000 unique listeners. High stream counts for certain artists may reflect intense fanbases (e.g., BTS ARMY, Swifties) as much as broad reach. This is especially worth noting if we compare artists by total streams.

**4. Geographic and language representation**  
Even "global" Spotify charts skew toward English-language and Latin markets. Artists from South Asia, Sub-Saharan Africa, East Asia (outside K-pop), and the Middle East are largely invisible here — not because their music isn't popular locally, but because Spotify's user base and recommendation algorithm have geographic density patterns. Any claim about "what people around the world listen to" from this data would be misleading.

**5. The recency effect**  
Older tracks in this dataset (released pre-2015) have had more calendar time to accumulate streams. Newer tracks (2022–2023) are competing on a platform with a larger user base but less accumulation time. Raw stream counts are not directly comparable across release years without this context.

---

### Could my charts mislead?

- A truncated y-axis on the streams chart would make small differences look enormous — must start at zero.  
- Sorting bars by stream count without noting the playlist-advantage context implies audio quality = streams.  
- Comparing danceability or energy across years implies Spotify's catalog changed — but really the *sample* changed (only hits are included each year).

---

### What disclaimers are needed?

Every visualization in this notebook should carry the understanding that:
> *This data represents only the most-streamed tracks on Spotify in 2023. Conclusions apply to this sample — not to music broadly, not to all listeners globally, and not to what makes music "good."*

---


## 3. Polished EDA Visuals

These are revised versions of standard exploratory charts, applying Week 4 principles:
- y-axes start at zero
- Single purposeful color (no rainbow decoration)  
- Descriptive, neutral titles  
- Axis labels with units  
- Data source cited on every chart  
- No chartjunk (no heavy grid lines, no hatching, no bold orange tick labels)


In [ ]:
SOURCE = "Source: Most Streamed Spotify Songs 2023 — Nidula Elgiriyewithana / Kaggle"
BLUE   = "#2563EB"
GRAY   = "#6B7280"

def add_source(ax, fig):
    fig.text(0.12, 0.01, SOURCE, fontsize=7.5, color=GRAY)

def clean_axes(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle="--", linewidth=0.55, alpha=0.45)


### Chart A — Distribution of Stream Counts

**Question:** How are stream counts distributed among the most-streamed songs?  
**Why this chart type:** A histogram shows the shape of a distribution — whether it's skewed, where most values cluster, and whether outliers exist.  
**Design choices:** Log x-axis because stream counts span several orders of magnitude; linear scale would compress 95% of songs into a thin sliver on the left. Neutral blue, no decoration.  
**Accuracy note:** This chart makes the survivorship bias visible — even within "most streamed," a tiny group of megahits dwarfs the rest.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df["streams"] / 1e9, bins=40, color=BLUE, edgecolor="white", linewidth=0.4)
ax.set_xlabel("Streams (billions)", fontsize=11)
ax.set_ylabel("Number of Tracks", fontsize=11)
ax.set_title("Stream Count Distribution — Most-Streamed Spotify Tracks, 2023", fontsize=13, pad=12)
clean_axes(ax)
add_source(ax, fig)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


### Chart B — Release Year of Most-Streamed Tracks

**Question:** When were the most-streamed songs released?  
**Why this chart type:** A bar chart — one bar per year — makes year-over-year comparisons easy to read.  
**Design choices:** Bars sorted chronologically (natural order for time data). Value labels on top of each bar for precision. y-axis starts at zero.  
**Accuracy note:** Recent years (2022–2023) appear dominant partly because of recency in the charts snapshot — these tracks had just been promoted. Older tracks that appear here have had years to accumulate; the comparison is not apples-to-apples.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

year_counts = df["released_year"].value_counts().sort_index()
bars = ax.bar(year_counts.index.astype(str), year_counts.values, color=BLUE, width=0.6, edgecolor="white")

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1.5, str(int(h)),
            ha="center", va="bottom", fontsize=8.5, color="#374151")

ax.set_xlabel("Release Year", fontsize=11)
ax.set_ylabel("Number of Tracks", fontsize=11)
ax.set_title("Release Year of Most-Streamed Spotify Tracks, 2023 Charts", fontsize=13, pad=12)
ax.set_ylim(0, year_counts.max() * 1.15)
clean_axes(ax)
add_source(ax, fig)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


### Chart C — Audio Feature Distributions (Box Plots)

**Question:** What is the typical range and spread of each audio feature across top tracks?  
**Why this chart type:** Box plots show median, spread (IQR), and outliers in one compact view — better than six separate histograms for comparison across features.  
**Design choices:** Features normalized to 0–100 scale (they already are); plotted side-by-side on one axis for direct visual comparison. Neutral color palette. No individual jitter (too many points).  
**Accuracy note:** These distributions describe only top-streamed hits — they do not represent all music on Spotify.


In [ ]:
features = ["danceability_%", "energy_%", "valence_%", "acousticness_%", "speechiness_%", "liveness_%"]
labels   = ["Danceability", "Energy", "Valence", "Acousticness", "Speechiness", "Liveness"]

# Rename for clean labels in plot
plot_df = df[features].copy()
plot_df.columns = labels

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(
    [plot_df[col].dropna() for col in labels],
    labels=labels,
    patch_artist=True,
    medianprops=dict(color="white", linewidth=2),
    whiskerprops=dict(color=GRAY),
    capprops=dict(color=GRAY),
    flierprops=dict(marker="o", markersize=3, alpha=0.3, color=GRAY)
)
for patch in bp["boxes"]:
    patch.set_facecolor(BLUE)
    patch.set_alpha(0.75)

ax.set_ylabel("Feature Value (0–100 scale)", fontsize=11)
ax.set_title("Audio Feature Distributions — Most-Streamed Spotify Tracks, 2023", fontsize=13, pad=12)
clean_axes(ax)
add_source(ax, fig)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


## 4. Polished Story Visuals

These charts are intentional — each one is built around a specific question and designed to communicate clearly to someone unfamiliar with the topic.


### Story Chart 1 — Does Playlist Presence Predict Streaming Success?

**Question this answers:** Is there a relationship between how many Spotify playlists a track appears in and how many streams it accumulates?

**Why this chart type:** A scatter plot with a trend line (LOBESS/regression) shows the direction and shape of a relationship between two continuous variables — something a bar chart or histogram cannot do.

**Design choices:** Semi-transparent points to reveal density where many tracks overlap. A single regression line in orange to indicate direction without implying a precise fit. Log scale on the y-axis (streams) because the distribution is heavily right-skewed. Axis labels include units.

**How accuracy was ensured:** y-axis starts at a natural minimum for log scale (no truncation). The trend line is drawn without overfitting. A note acknowledges that correlation is not causation — playlist placement may drive streams, *or* hit songs may get added to more playlists after the fact, or both.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = df["in_spotify_playlists"].dropna()
y = df.loc[x.index, "streams"]
mask = (x > 0) & (y > 0)
x, y = x[mask], y[mask]

ax.scatter(x, np.log10(y), alpha=0.25, s=18, color=BLUE, linewidths=0)

# Simple linear trend on log10(streams)
z = np.polyfit(x, np.log10(y), 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.quantile(0.98), 200)
ax.plot(x_line, p(x_line), color="#F97316", linewidth=2, label="Linear trend")

ax.set_xlabel("Number of Spotify Playlists Track Appears In", fontsize=11)
ax.set_ylabel("Streams (log₁₀ scale)", fontsize=11)
ax.set_title("Playlist Presence vs. Streaming Volume — Spotify Top Tracks, 2023", fontsize=13, pad=12)

# Readable y-tick labels
yticks = [6, 7, 8, 9, 10]
ax.set_yticks(yticks)
ax.set_yticklabels([f"10^{t}" for t in yticks])

ax.legend(fontsize=9, frameon=False)
clean_axes(ax)
ax.grid(axis="both", linestyle="--", linewidth=0.45, alpha=0.4)

fig.text(0.12, 0.01, SOURCE, fontsize=7.5, color=GRAY)
fig.text(0.12, -0.03,
         "⚠ Correlation ≠ causation. Playlist placement may drive streams, or popular tracks may earn more placements — or both.",
         fontsize=7.5, color="#B45309")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()


### Story Chart 2 — Solo vs. Collaboration: Who Streams More?

**Question this answers:** Do tracks with multiple credited artists tend to accumulate more streams than solo tracks?

**Why this chart type:** A grouped box plot makes the median and spread of each group directly comparable. A simple bar chart showing only averages would hide the wide variance within each group.

**Design choices:** Grouped by artist count, capped at 3+ collaborators (very few tracks have more). Neutral color, no decoration. Median line clearly visible. Outlier points shown but small and semi-transparent to avoid overplotting.

**How accuracy was ensured:** y-axis starts at zero (streams in billions). Groups with fewer than 10 tracks are noted. No claim is made about causation — this only describes patterns in this sample of already-successful tracks.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

df["artist_group"] = df["artist_count"].apply(
    lambda x: "Solo" if x == 1 else ("2 Artists" if x == 2 else "3+ Artists")
)
order = ["Solo", "2 Artists", "3+ Artists"]
group_data = [df.loc[df["artist_group"] == g, "streams_B"].dropna() for g in order]
counts = [len(g) for g in group_data]

bp = ax.boxplot(
    group_data,
    labels=[f"{g}
(n={c})" for g, c in zip(order, counts)],
    patch_artist=True,
    medianprops=dict(color="white", linewidth=2.5),
    whiskerprops=dict(color=GRAY),
    capprops=dict(color=GRAY),
    flierprops=dict(marker="o", markersize=4, alpha=0.3, color=GRAY)
)
for patch in bp["boxes"]:
    patch.set_facecolor(BLUE)
    patch.set_alpha(0.75)

ax.set_ylabel("Streams (billions)", fontsize=11)
ax.set_title("Stream Counts by Number of Credited Artists — Spotify Top Tracks, 2023", fontsize=13, pad=12)
ax.set_ylim(0, df["streams_B"].quantile(0.99) * 1.15)
clean_axes(ax)
add_source(ax, fig)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


### Story Chart 3 — Do Audio Features Shift by Release Era?

**Question this answers:** Have the audio characteristics of hit songs changed over time? Specifically, are newer top-streaming tracks more or less danceable and energetic than older ones?

**Why this chart type:** A line chart with shaded confidence intervals shows trends over time more clearly than a bar chart, and the shading communicates uncertainty around the mean.

**Design choices:** Two lines on one chart (danceability and energy) so they can be directly compared across eras. Shaded band = ±1 standard deviation, not a margin of error, and labeled as such. Years with fewer than 5 tracks are excluded to avoid noisy estimates from small samples.

**Accuracy / ethics note:** Remember — this is not a random sample of all music. Songs from 2015 in this dataset are the ones that *still rank among the most-streamed in 2023*, which may not be representative of the typical hit from that year.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

era = df[df["released_year"] >= 2010].copy()
yearly = era.groupby("released_year").agg(
    dance_mean=("danceability_%", "mean"),
    dance_std=("danceability_%", "std"),
    energy_mean=("energy_%", "mean"),
    energy_std=("energy_%", "std"),
    n=("streams", "count")
).reset_index()

# Only years with enough tracks to be meaningful
yearly = yearly[yearly["n"] >= 5]

ax.plot(yearly["released_year"], yearly["dance_mean"], color=BLUE, linewidth=2.5, marker="o", markersize=5, label="Danceability")
ax.fill_between(yearly["released_year"],
                yearly["dance_mean"] - yearly["dance_std"],
                yearly["dance_mean"] + yearly["dance_std"],
                alpha=0.12, color=BLUE)

ax.plot(yearly["released_year"], yearly["energy_mean"], color="#F97316", linewidth=2.5, marker="s", markersize=5, label="Energy")
ax.fill_between(yearly["released_year"],
                yearly["energy_mean"] - yearly["energy_std"],
                yearly["energy_mean"] + yearly["energy_std"],
                alpha=0.12, color="#F97316")

ax.set_xlabel("Track Release Year", fontsize=11)
ax.set_ylabel("Mean Feature Value (0–100)", fontsize=11)
ax.set_title("Danceability & Energy of Top-Streamed Hits by Release Year", fontsize=13, pad=12)
ax.set_ylim(0, 100)
ax.legend(fontsize=10, frameon=False)
clean_axes(ax)

fig.text(0.12, 0.01, SOURCE, fontsize=7.5, color=GRAY)
fig.text(0.12, -0.025,
         "Shaded band = ±1 standard deviation. Only release years with ≥5 tracks included.",
         fontsize=7.5, color=GRAY)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()


## 5. Repository Notes & TODOs

**Completed this week:**
- [x] Dataset swapped to Most Streamed Spotify Songs 2023 (better outcome variable, richer merge potential)
- [x] Ethics lens section written — survivorship bias, playlist advantage, geographic gaps, streams ≠ listeners documented
- [x] Three polished EDA charts (distribution, release year, box plot features)
- [x] Three story charts with written explanations (playlist vs. streams, solo vs. collab, audio features over time)

**Ideas to come back to:**
- [ ] **Merge with Billboard Hot 100** — match by track name + artist to see overlap/divergence between Spotify and radio charts; some tracks dominate Spotify but barely touch Billboard and vice versa
- [ ] **Key and mode analysis** — does major vs. minor key affect stream counts? Box plot by mode (Major/Minor)
- [ ] **Release month seasonality** — do tracks released in certain months accumulate more streams? Could use a polar/radial bar chart
- [ ] **Top artists deep dive** — filter to top 20 artists by total streams, look at their audio feature fingerprints side by side
- [ ] **Cross-platform comparison** — Spotify playlist count vs. Apple playlist count vs. Deezer: which platform's curation correlates most strongly with Spotify streams?
- [ ] Clean up any remaining exploratory scratch cells from earlier weeks before final submission
